# Train Flatmate RL with TRL GRPO

This notebook uses the live Hugging Face Space endpoint as the reward source for GRPO. It first collects prompt states from websocket rollouts, then trains a model to generate one JSON action for each observation. The main reward function resets the endpoint, replays the stored action prefix, applies the model action, and scores the resulting Flatmate RL transition.

Space endpoint: `https://huggingface.co/spaces/kushalExplores/flatmate_rl`

In [ ]:
# Restart the kernel after this cell if your notebook runtime asks you to.
%pip install -q "trl>=0.26.2" "transformers>=4.57.0" accelerate datasets peft websockets huggingface_hub matplotlib pandas

In [ ]:
from __future__ import annotations

import asyncio
import json
import random
import threading
from dataclasses import dataclass
from typing import Any
from urllib.parse import urlparse

import websockets
from datasets import Dataset

SPACE_HTTP_URL = "https://kushalexplores-flatmate-rl.hf.space"
SCENARIOS = [
    "task_visit_single",
    "task_visit_single_hidden_flex",
    "task_visit_multi",
    "task_visit_single_seller_followup",
]

def ws_url_from_http(base_url: str) -> str:
    parsed = urlparse(base_url.rstrip("/"))
    scheme = "wss" if parsed.scheme == "https" else "ws"
    return f"{scheme}://{parsed.netloc}/ws"

SPACE_WS_URL = ws_url_from_http(SPACE_HTTP_URL)
SPACE_WS_URL

## Websocket Environment Client

OpenEnv keeps episode state on `/ws`. The plain HTTP `/reset` and `/step` endpoints create fresh environment instances, so GRPO reward replay uses websocket sessions.

In [ ]:
class FlatmateEndpoint:
    def __init__(self, ws_url: str = SPACE_WS_URL, timeout_s: float = 120.0):
        self.ws_url = ws_url
        self.timeout_s = timeout_s

    async def __aenter__(self):
        self.ws = await websockets.connect(
            self.ws_url,
            open_timeout=self.timeout_s,
            ping_timeout=self.timeout_s,
        )
        return self

    async def __aexit__(self, exc_type, exc, tb):
        try:
            await self.ws.send(json.dumps({"type": "close"}))
        finally:
            await self.ws.close()

    async def _send(self, payload: dict[str, Any]) -> dict[str, Any]:
        await self.ws.send(json.dumps(payload))
        raw = await asyncio.wait_for(self.ws.recv(), timeout=self.timeout_s)
        message = json.loads(raw)
        if message.get("type") == "error":
            raise RuntimeError(message.get("data", message))
        data = message["data"]
        obs = data.get("observation", {})
        obs["reward"] = data.get("reward")
        obs["done"] = data.get("done", False)
        return obs

    async def reset(self, scenario_id: str, seed: int | None = None) -> dict[str, Any]:
        data: dict[str, Any] = {"scenario_id": scenario_id}
        if seed is not None:
            data["seed"] = seed
        return await self._send({"type": "reset", "data": data})

    async def step(self, action: dict[str, Any]) -> dict[str, Any]:
        return await self._send({"type": "step", "data": action})

async def smoke_test_endpoint():
    async with FlatmateEndpoint() as env:
        obs = await env.reset("task_visit_single", seed=1)
        print(obs["scenario_id"], obs["status"])
        print(obs.get("last_user_message") or obs.get("current_user_request"))

await smoke_test_endpoint()

## Prompt States

GRPO needs prompts. These prompts are endpoint observations collected from heuristic rollouts. Each row also stores the action prefix needed to reconstruct that exact state during reward scoring.

In [ ]:
def trace_tool_names(obs: dict[str, Any]) -> list[str]:
    return [str(t.get("tool", t.get("tool_name", ""))) for t in obs.get("tool_trace", [])]

def heuristic_action(obs: dict[str, Any]) -> dict[str, Any] | None:
    """Per-scenario heuristic mirroring the server's autopolicy_next_request.

    The previous version hard-coded post_031 / "tomorrow 7pm" for every
    scenario, which caused calendar_mismatch and consent_violation in the env
    (book_viewing requires that (post_id, time_text) was returned by
    check_calendar_slots AND that BOTH buyer and poster confirmed that slot).
    """
    if obs.get("done"):
        return None

    tool_names = trace_tool_names(obs)
    phase = obs.get("phase", "buyer")
    remaining = set(obs.get("remaining_required_fields", []))
    scenario_id = obs.get("scenario_id", "task_visit_single")
    booked = [str(item.get("post_id", "")) for item in obs.get("booked_visits", [])]
    selected_posts = list(obs.get("selected_posts", []))
    last_user_message = str(obs.get("last_user_message", "")).lower()
    buyer_history = obs.get("buyer_conversation_history", [])
    last_role = str(buyer_history[-1].get("role", "")) if buyer_history else ""
    user_has_replied = obs.get("status") == "user_response" and last_role == "user"

    def has(name: str) -> bool:
        return name in tool_names

    def count(name: str) -> int:
        return tool_names.count(name)

    def ask_for_buyer_fields() -> dict[str, Any] | None:
        if "diet" in remaining and "visit_availability" in remaining:
            return {"action_type": "assistant_message", "assistant_message": "Please share your dietary preference and visit availability."}
        if "diet" in remaining:
            return {"action_type": "assistant_message", "assistant_message": "Please share your dietary preference."}
        if "visit_availability" in remaining:
            return {"action_type": "assistant_message", "assistant_message": "Please share your visit availability."}
        return None

    def ask_for_seller_fields() -> dict[str, Any] | None:
        need_dietary = "dietary" in remaining
        need_occupation = "occupation_requirement" in remaining
        need_slots = "calendar_slots" in remaining
        if need_dietary and need_occupation and need_slots:
            return {"action_type": "assistant_message", "assistant_message": "Please share the household dietary setup, who the flat is for, and available visit time slots."}
        if need_dietary or need_occupation or need_slots:
            return {"action_type": "assistant_message", "assistant_message": "Please share the remaining seller details (dietary setup, who the flat is for, available visit time slots)."}
        return None

    if phase == "seller":
        if not obs.get("seller_profile_stored"):
            ask = ask_for_seller_fields()
            if ask is not None:
                return ask
            return {"action_type": "tool_call", "tool_name": "store_seller_details", "tool_arguments": {}}
        if not has("match_location_preference"):
            return {"action_type": "tool_call", "tool_name": "match_location_preference", "tool_arguments": {"post_ids": ["post_dynamic_followup_1"]}}
        if not has("check_table_slot_matches"):
            return {"action_type": "tool_call", "tool_name": "check_table_slot_matches", "tool_arguments": {"post_ids": ["post_dynamic_followup_1"]}}
        if not has("confirm_seller_match"):
            return {"action_type": "tool_call", "tool_name": "confirm_seller_match", "tool_arguments": {"post_id": "post_dynamic_followup_1", "time_text": "Sunday 5pm"}}
        if not has("offer_matched_listing_to_buyer"):
            return {"action_type": "tool_call", "tool_name": "offer_matched_listing_to_buyer", "tool_arguments": {"post_id": "post_dynamic_followup_1", "time_text": "Sunday 5pm"}}
        return {"action_type": "tool_call", "tool_name": "schedule_table_visit", "tool_arguments": {"post_id": "post_dynamic_followup_1", "time_text": "Sunday 5pm"}}

    # task_visit_single_seller_followup: buyer phase ends with close_buyer_conversation
    if scenario_id == "task_visit_single_seller_followup" and not obs.get("seller_profile_stored"):
        if not obs.get("buyer_profile_stored"):
            ask = ask_for_buyer_fields()
            if ask is not None:
                return ask
            return {"action_type": "tool_call", "tool_name": "store_user_details", "tool_arguments": {}}
        if not has("search_posts"):
            return {"action_type": "tool_call", "tool_name": "search_posts", "tool_arguments": {}}
        return {"action_type": "tool_call", "tool_name": "close_buyer_conversation", "tool_arguments": {}}

    if not obs.get("buyer_profile_stored"):
        ask = ask_for_buyer_fields()
        if ask is not None:
            return ask
        return {"action_type": "tool_call", "tool_name": "store_user_details", "tool_arguments": {}}

    if not has("search_posts"):
        return {"action_type": "tool_call", "tool_name": "search_posts", "tool_arguments": {}}

    if scenario_id == "task_visit_single":
        if not has("match_location_preference"):
            return {"action_type": "tool_call", "tool_name": "match_location_preference", "tool_arguments": {"post_ids": ["post_023", "post_031"]}}
        if not has("get_commute_time"):
            return {"action_type": "tool_call", "tool_name": "get_commute_time", "tool_arguments": {"post_ids": ["post_023", "post_031"]}}
        if not has("check_calendar_slots"):
            return {"action_type": "tool_call", "tool_name": "check_calendar_slots", "tool_arguments": {"post_ids": ["post_023"]}}
        if not has("contact_poster") and not has("book_viewing") and not user_has_replied:
            return {"action_type": "assistant_message", "assistant_message": "post_023 is available Saturday 11am. Please confirm Saturday 11am if that works."}
        if not has("contact_poster"):
            return {"action_type": "tool_call", "tool_name": "contact_poster", "tool_arguments": {"post_id": "post_023", "time_text": "Saturday 11am"}}
        if not has("book_viewing"):
            return {"action_type": "tool_call", "tool_name": "book_viewing", "tool_arguments": {"post_id": "post_023", "time_text": "Saturday 11am"}}
        return None

    if scenario_id == "task_visit_single_hidden_flex":
        if not has("match_location_preference"):
            return {"action_type": "tool_call", "tool_name": "match_location_preference", "tool_arguments": {"post_ids": ["post_023", "post_052"]}}
        if not has("get_commute_time"):
            return {"action_type": "tool_call", "tool_name": "get_commute_time", "tool_arguments": {"post_ids": ["post_023", "post_052"]}}
        if not has("check_calendar_slots"):
            return {"action_type": "tool_call", "tool_name": "check_calendar_slots", "tool_arguments": {"post_ids": ["post_023", "post_052"]}}
        if not has("contact_poster") and not has("book_viewing") and not user_has_replied:
            return {"action_type": "assistant_message", "assistant_message": "No Tuesday slot matches. I can offer Saturday 1pm or Sunday 5pm instead."}
        if not has("contact_poster"):
            return {"action_type": "tool_call", "tool_name": "contact_poster", "tool_arguments": {"post_id": "post_023", "time_text": "Sunday 5pm"}}
        if not has("book_viewing"):
            return {"action_type": "tool_call", "tool_name": "book_viewing", "tool_arguments": {"post_id": "post_023", "time_text": "Sunday 5pm"}}
        return None

    if scenario_id == "task_visit_multi":
        if not has("match_location_preference"):
            return {"action_type": "tool_call", "tool_name": "match_location_preference", "tool_arguments": {"post_ids": ["post_031", "post_052"]}}
        if not has("get_commute_time"):
            return {"action_type": "tool_call", "tool_name": "get_commute_time", "tool_arguments": {"post_ids": ["post_031", "post_052"]}}
        if not has("check_calendar_slots"):
            return {"action_type": "tool_call", "tool_name": "check_calendar_slots", "tool_arguments": {"post_ids": ["post_031", "post_052"]}}
        if not selected_posts and not user_has_replied:
            return {"action_type": "assistant_message", "assistant_message": "I shortlisted post_031 at tomorrow 7pm and post_052 at Sunday 4pm. Which listings do you want to pursue?"}
        # First booking: post_031 / tomorrow 7pm
        if "post_031" not in booked and count("contact_poster") == 0 and "confirm tomorrow 7pm" not in last_user_message:
            return {"action_type": "assistant_message", "assistant_message": "Please confirm tomorrow 7pm for post_031."}
        if "post_031" not in booked and count("contact_poster") == 0:
            return {"action_type": "tool_call", "tool_name": "contact_poster", "tool_arguments": {"post_id": "post_031", "time_text": "tomorrow 7pm"}}
        if "post_031" not in booked:
            return {"action_type": "tool_call", "tool_name": "book_viewing", "tool_arguments": {"post_id": "post_031", "time_text": "tomorrow 7pm"}}
        # Second booking: post_052 / Sunday 4pm
        if "post_052" not in booked and count("contact_poster") == 1 and "confirm sunday 4pm" not in last_user_message:
            return {"action_type": "assistant_message", "assistant_message": "Please confirm Sunday 4pm for post_052."}
        if "post_052" not in booked and count("contact_poster") == 1 and count("book_viewing") == 1:
            return {"action_type": "tool_call", "tool_name": "contact_poster", "tool_arguments": {"post_id": "post_052", "time_text": "Sunday 4pm"}}
        if "post_052" not in booked:
            return {"action_type": "tool_call", "tool_name": "book_viewing", "tool_arguments": {"post_id": "post_052", "time_text": "Sunday 4pm"}}
        return None

    return None

def compact_observation(obs: dict[str, Any]) -> dict[str, Any]:
    return {
        "scenario_id": obs.get("scenario_id"),
        "phase": obs.get("phase"),
        "status": obs.get("status"),
        "buyer_profile_stored": obs.get("buyer_profile_stored", False),
        "seller_profile_stored": obs.get("seller_profile_stored", False),
        "last_user_message": obs.get("last_user_message"),
        "current_user_request": obs.get("current_user_request"),
        "available_tools": obs.get("available_tools", []),
        "remaining_required_fields": obs.get("remaining_required_fields", []),
        "prerequisites_satisfied": obs.get("prerequisites_satisfied", {}),
        "recent_tool_calls": obs.get("recent_tool_calls", []),
        "last_tool_result": obs.get("last_tool_result", {}),
        "selected_posts": obs.get("selected_posts", []),
        "violations": obs.get("violations", []),
        "booked_visits": obs.get("booked_visits", []),
        "feedback_summary": obs.get("feedback_summary", ""),
    }

def prompt_from_observation(obs: dict[str, Any]) -> str:
    return (
        "You are a broker policy for the Flatmate RL environment. "
        "Return exactly one JSON action and no extra text.\n\n"
        "Valid action shapes:\n"
        "{\"action_type\":\"assistant_message\",\"assistant_message\":\"...\"}\n"
        "{\"action_type\":\"tool_call\",\"tool_name\":\"...\",\"tool_arguments\":{...}}\n\n"
        f"Observation:\n{json.dumps(compact_observation(obs), ensure_ascii=False, sort_keys=True)}\n\n"
        "Action:\n"
    )

In [ ]:
@dataclass
class PromptCollectionConfig:
    episodes: int = 12
    max_steps: int = 14
    seed: int = 11

async def collect_prompt_rows(config: PromptCollectionConfig = PromptCollectionConfig()) -> list[dict[str, Any]]:
    rng = random.Random(config.seed)
    rows: list[dict[str, Any]] = []
    for episode_idx in range(config.episodes):
        scenario_id = rng.choice(SCENARIOS)
        prefix_actions: list[dict[str, Any]] = []
        async with FlatmateEndpoint() as env:
            obs = await env.reset(scenario_id, seed=config.seed + episode_idx)
            for step_idx in range(config.max_steps):
                action = heuristic_action(obs)
                if action is None or obs.get("done"):
                    break
                rows.append({
                    "prompt": prompt_from_observation(obs),
                    "scenario_id": scenario_id,
                    "seed": config.seed + episode_idx,
                    "prefix_actions_json": json.dumps(prefix_actions, ensure_ascii=False),
                    "reference_action_json": json.dumps(action, ensure_ascii=False, sort_keys=True),
                })
                obs = await env.step(action)
                prefix_actions.append(action)
                if obs.get("done"):
                    break
        print(f"episode={episode_idx:02d} scenario={scenario_id} total_rows={len(rows)}")
    return rows

rows = await collect_prompt_rows(PromptCollectionConfig(episodes=12, max_steps=14))

# GRPO and SFT both need chat-format prompts so the model sees the Qwen
# <|im_start|>user / <|im_start|>assistant template it was pretrained with.
# Without this the model confuses the raw "Action:\n{" suffix with pretraining
# chat formats like "{Human: ...}" and generates garbage instead of JSON.
train_dataset = Dataset.from_list([
    {
        **row,
        "prompt": [{"role": "user", "content": row["prompt"]}],
    }
    for row in rows
])
train_dataset

## GRPO Rewards

`json_format_reward` is cheap and runs for every completion. `endpoint_transition_reward` is slower because it calls the live Space: it replays the prefix actions, sends the generated action, and returns a reward based on environment reward, validity, violations, and bookings.

In [ ]:
def completion_text(completion: Any) -> str:
    if isinstance(completion, str):
        return completion
    if isinstance(completion, list) and completion and isinstance(completion[0], dict):
        return str(completion[0].get("content", ""))
    return str(completion)

def parse_json_action(text: str) -> dict[str, Any]:
    text = completion_text(text).strip()
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError("completion does not contain a JSON object")
    action = json.loads(text[start : end + 1])
    if action.get("action_type") == "assistant_message":
        if not str(action.get("assistant_message", "")).strip():
            raise ValueError("assistant_message action needs assistant_message")
        return {
            "action_type": "assistant_message",
            "assistant_message": str(action["assistant_message"]),
        }
    if action.get("action_type") == "tool_call":
        if not str(action.get("tool_name", "")).strip():
            raise ValueError("tool_call action needs tool_name")
        args = action.get("tool_arguments", {})
        if not isinstance(args, dict):
            raise ValueError("tool_arguments must be an object")
        return {
            "action_type": "tool_call",
            "tool_name": str(action["tool_name"]),
            "tool_arguments": args,
        }
    raise ValueError("action_type must be assistant_message or tool_call")

REWARD_DEBUG = {"parse_errors": [], "endpoint_errors": [], "prefix_done": 0, "valid": 0, "invalid": 0}

def json_format_reward(completions, **kwargs) -> list[float]:
    rewards = []
    for completion in completions:
        try:
            parse_json_action(completion_text(completion))
            rewards.append(0.5)
            REWARD_DEBUG["valid"] += 1
        except Exception as exc:
            rewards.append(-1.0)
            REWARD_DEBUG["invalid"] += 1
            if len(REWARD_DEBUG["parse_errors"]) < 20:
                REWARD_DEBUG["parse_errors"].append({"error": str(exc), "completion": completion_text(completion)[:240]})
    return rewards

async def score_one_completion(
    completion: Any,
    scenario_id: str,
    seed: int,
    prefix_actions_json: str,
) -> float:
    try:
        action = parse_json_action(completion_text(completion))
        prefix_actions = json.loads(prefix_actions_json)
    except Exception:
        return -1.0

    try:
        async with FlatmateEndpoint() as env:
            obs = await env.reset(scenario_id, seed=int(seed))
            for prefix_action in prefix_actions:
                obs = await env.step(prefix_action)
                if obs.get("done"):
                    # Prefix already terminated the env; we cannot score the model
                    # action here. Use a small neutral penalty so this prompt
                    # contributes near-zero advantage to the GRPO group.
                    REWARD_DEBUG["prefix_done"] += 1
                    return -0.1

            before_violations = len(obs.get("violations", []))
            before_bookings = len(obs.get("booked_visits", []))
            obs = await env.step(action)

        reward = float(obs.get("reward") or obs.get("step_reward") or 0.0)
        reward += 0.15
        if len(obs.get("violations", [])) > before_violations:
            reward -= 0.75
        if len(obs.get("booked_visits", [])) > before_bookings:
            reward += 1.0
        if obs.get("done") and obs.get("status") != "failed":
            reward += 0.5
        return float(max(-2.0, min(2.0, reward)))
    except Exception as exc:
        if len(REWARD_DEBUG["endpoint_errors"]) < 10:
            REWARD_DEBUG["endpoint_errors"].append(repr(exc))
        return -1.0

async def score_completion_batch(completions, scenario_id, seed, prefix_actions_json) -> list[float]:
    tasks = [
        score_one_completion(c, s, int(sd), p)
        for c, s, sd, p in zip(completions, scenario_id, seed, prefix_actions_json)
    ]
    return list(await asyncio.gather(*tasks))

def run_async_blocking(coro):
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)

    result: dict[str, Any] = {}
    def runner():
        try:
            result["value"] = asyncio.run(coro)
        except Exception as exc:
            result["error"] = exc

    thread = threading.Thread(target=runner, daemon=True)
    thread.start()
    thread.join()
    if "error" in result:
        raise result["error"]
    return result["value"]

def endpoint_transition_reward(completions, scenario_id, seed, prefix_actions_json, **kwargs) -> list[float]:
    return run_async_blocking(score_completion_batch(completions, scenario_id, seed, prefix_actions_json))

# Quick reward smoke test with known reference actions from the collected dataset.
sample = train_dataset.select(range(min(2, len(train_dataset))))
endpoint_transition_reward(
    completions=sample["reference_action_json"],
    scenario_id=sample["scenario_id"],
    seed=sample["seed"],
    prefix_actions_json=sample["prefix_actions_json"],
)

## SFT Warmup

Before running GRPO, do a short supervised warmup on the heuristic's `(prompt, reference_action_json)` pairs. This gives the LoRA adapter a strong prior for emitting valid JSON in the expected schema. Without it, GRPO on a 0.5B model with a sparse reward signal collapses the model and every completion becomes unparseable.

In [ ]:
import inspect

import torch
from peft import LoraConfig, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_DIR = "flatmate-rl-grpo-policy"
SFT_OUTPUT_DIR = f"{OUTPUT_DIR}/sft_warmup"

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

# Conversational format: SFTTrainer applies the Qwen chat template automatically
# so the model learns to produce JSON inside the <|im_start|>assistant role,
# exactly matching what eval and GRPO see at inference time.
sft_dataset = Dataset.from_list([
    {
        "prompt": [{"role": "user", "content": row["prompt"]}],
        "completion": [{"role": "assistant", "content": row["reference_action_json"]}],
    }
    for row in rows
])
print(f"SFT dataset size: {len(sft_dataset)}")
print(f"Example completion: {sft_dataset[0]['completion'][0]['content']!r}")

def make_sft_config(**kwargs):
    valid = set(inspect.signature(SFTConfig.__init__).parameters)
    filtered = {key: value for key, value in kwargs.items() if key in valid}
    skipped = sorted(set(kwargs) - set(filtered))
    if skipped:
        print("Skipping unsupported SFTConfig fields for this TRL version:", skipped)
    return SFTConfig(**filtered)

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
sft_args = make_sft_config(
    output_dir=SFT_OUTPUT_DIR,
    num_train_epochs=4,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=2,
    save_strategy="epoch",
    save_total_limit=1,
    bf16=use_bf16,
    fp16=torch.cuda.is_available() and not use_bf16,
    max_length=1700,
    completion_only_loss=True,
    report_to="none",
)

sft_trainer = SFTTrainer(
    model=MODEL_NAME,
    args=sft_args,
    train_dataset=sft_dataset,
    peft_config=peft_config,
)
sft_trainer.train()
sft_trainer.save_model(SFT_OUTPUT_DIR)
print(f"\nSFT warmup saved to {SFT_OUTPUT_DIR}")
print("SFT loss history (last 5):")
for entry in sft_trainer.state.log_history[-5:]:
    print(" ", {k: v for k, v in entry.items() if k in {"step", "loss", "epoch", "train_loss"}})

# ── Sanity check: does the SFT model emit valid JSON and stop before max_new_tokens? ──
_sft_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if _sft_tokenizer.pad_token is None:
    _sft_tokenizer.pad_token = _sft_tokenizer.eos_token
_sft_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16 if use_bf16 else torch.float32, device_map="auto",
)
_sft_model = PeftModel.from_pretrained(_sft_base, SFT_OUTPUT_DIR)
_sft_model.eval()

print("\n=== SFT sanity check (using chat template) ===")
_n_terminated = 0
for _i, _row in enumerate(rows[:4]):
    _chat = _sft_tokenizer.apply_chat_template(
        [{"role": "user", "content": _row["prompt"]}],
        tokenize=False, add_generation_prompt=True,
    )
    _inputs = _sft_tokenizer(_chat, return_tensors="pt", truncation=True, max_length=1700).to(_sft_model.device)
    with torch.no_grad():
        _out = _sft_model.generate(
            **_inputs, max_new_tokens=200, do_sample=False,
            eos_token_id=_sft_tokenizer.eos_token_id,
            pad_token_id=_sft_tokenizer.eos_token_id,
        )
    _n_in = _inputs["input_ids"].shape[-1]
    _n_gen = _out.shape[-1] - _n_in
    _terminated = bool((_out[0][_n_in:] == _sft_tokenizer.eos_token_id).any())
    _n_terminated += int(_terminated)
    _completion = _sft_tokenizer.decode(_out[0][_n_in:], skip_special_tokens=True)
    print(f"  [{_i}] gen_tokens={_n_gen:3d}  terminated_with_eos={_terminated}")
    print(f"       completion : {_completion[:120]!r}")
    print(f"       reference  : {_row['reference_action_json'][:120]!r}")

del _sft_base, _sft_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

if _n_terminated == 0:
    raise RuntimeError(
        "\n\nSFT sanity FAILED: 0/4 completions ended with EOS.\n"
        "This means the SFT model never learned to stop after the JSON object.\n"
        "GRPO will get clipped_ratio=1.0 and zero gradient again.\n"
        "Fix: check the 'Skipping unsupported SFTConfig fields' output above.\n"
        "If 'completion_only_loss' was skipped, your TRL version doesn't support it;\n"
        "upgrade TRL: pip install -q 'trl>=0.26.2' and restart the kernel."
    )
print(f"\nSFT sanity PASSED: {_n_terminated}/4 terminated with EOS. Proceeding to GRPO.")

## Train with GRPO

The endpoint reward is network-bound, so keep `num_generations`, dataset size, and max steps small until the loop is stable. Increase them once you see valid JSON actions and non-negative endpoint rewards.

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM
from trl import GRPOConfig, GRPOTrainer

# Load the base model and merge the SFT LoRA so GRPO starts from the
# warmed-up policy. Then attach a fresh LoRA adapter for GRPO updates so
# we can compute KL against the merged-base reference model implicitly.
sft_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16 if use_bf16 else torch.float32)
sft_warmed = PeftModel.from_pretrained(sft_base, SFT_OUTPUT_DIR)
sft_warmed = sft_warmed.merge_and_unload()
print("Loaded SFT-warmed base for GRPO from", SFT_OUTPUT_DIR)

def make_grpo_config(**kwargs):
    valid = set(inspect.signature(GRPOConfig.__init__).parameters)
    filtered = {key: value for key, value in kwargs.items() if key in valid}
    skipped = sorted(set(kwargs) - set(filtered))
    if skipped:
        print("Skipping unsupported GRPOConfig fields for this TRL version:", skipped)
    return GRPOConfig(**filtered)

training_args = make_grpo_config(
    output_dir=OUTPUT_DIR,
    learning_rate=5e-6,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_prompt_length=1700,
    max_completion_length=200,
    max_steps=60,
    logging_steps=1,
    save_steps=20,
    save_total_limit=2,
    beta=0.1,
    temperature=0.9,
    top_p=0.95,
    report_to="none",
    bf16=use_bf16,
    fp16=torch.cuda.is_available() and not use_bf16,
)

_grpo_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if _grpo_tokenizer.pad_token is None:
    _grpo_tokenizer.pad_token = _grpo_tokenizer.eos_token

trainer = GRPOTrainer(
    model=sft_warmed,
    processing_class=_grpo_tokenizer,
    reward_funcs=[json_format_reward, endpoint_transition_reward],
    args=training_args,
    train_dataset=train_dataset,
    peft_config=peft_config,
)

REWARD_DEBUG.update({"parse_errors": [], "endpoint_errors": [], "prefix_done": 0, "valid": 0, "invalid": 0})
train_result = trainer.train()
train_log_history = trainer.state.log_history
trainer.save_model(OUTPUT_DIR)
print("\nReward debug summary:")
print(f"  valid JSON completions:   {REWARD_DEBUG['valid']}")
print(f"  invalid JSON completions: {REWARD_DEBUG['invalid']}")
print(f"  prefix-done short circuits: {REWARD_DEBUG['prefix_done']}")
if REWARD_DEBUG["parse_errors"]:
    print("  first 3 parse errors:")
    for record in REWARD_DEBUG["parse_errors"][:3]:
        print("   -", record["error"], "::", record["completion"][:120].replace("\n", " "))
if REWARD_DEBUG["endpoint_errors"]:
    print("  first 3 endpoint errors:")
    for err in REWARD_DEBUG["endpoint_errors"][:3]:
        print("   -", err)
train_result

## Training Log

Plot logged GRPO reward and loss curves over optimizer steps.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

log_path = Path(OUTPUT_DIR) / "train_log_history.json"
log_path.parent.mkdir(parents=True, exist_ok=True)
log_path.write_text(json.dumps(train_log_history, indent=2))

def plot_training_log(log_history, title: str = "GRPO training log"):
    df = pd.DataFrame(log_history)
    if df.empty:
        print("No trainer log rows found yet.")
        return df
    metric_cols = [col for col in ["loss", "reward", "rewards/json_format_reward", "rewards/endpoint_transition_reward", "kl"] if col in df.columns]
    if not metric_cols:
        metric_cols = [col for col in df.columns if "reward" in col or col in {"loss", "kl"}]
    if not metric_cols or "step" not in df.columns:
        print("No plottable step metrics found. Available columns:", list(df.columns))
        return df
    axes = df.dropna(subset=["step"]).plot(
        x="step",
        y=metric_cols,
        marker="o",
        figsize=(9, 4),
        title=title,
    )
    axes.set_xlabel("optimizer step")
    axes.grid(True, alpha=0.3)
    plt.show()
    return df

train_log_df = plot_training_log(train_log_history)
train_log_df.tail()

## Evaluate One Episode

In [ ]:
import torch
from peft import AutoPeftModelForCausalLM
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model_for_eval = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    device_map="auto",
)
base_model_for_eval.eval()
base_model_for_eval.config.use_cache = False

loaded_model_for_eval = AutoPeftModelForCausalLM.from_pretrained(OUTPUT_DIR, device_map="auto")
loaded_model_for_eval.eval()
loaded_model_for_eval.config.use_cache = False
active_model = loaded_model_for_eval
print(f"Loaded base model from {MODEL_NAME}")
print(f"Loaded saved GRPO model from {OUTPUT_DIR}")

generation_stats = {"fallbacks": 0, "parse_errors": []}


def _build_inference_inputs(obs: dict[str, Any]):
    """Apply Qwen's chat template then tokenize.

    SFT and GRPO both used the chat template (conversational dataset format),
    so inference must use the same template. Without it the model sees a raw
    'Action:\\n' suffix and hallucinates '{Human: ...}' instead of valid JSON.
    """
    chat_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt_from_observation(obs)}],
        tokenize=False,
        add_generation_prompt=True,
    )
    return tokenizer(chat_text, return_tensors="pt", truncation=True, max_length=1700).to(active_model.device)


def _greedy_generate(inputs):
    active_model.generation_config.do_sample = False
    active_model.generation_config.temperature = None
    active_model.generation_config.top_p = None
    active_model.generation_config.top_k = None
    with torch.no_grad():
        return active_model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )


def safe_model_action(obs: dict[str, Any], log_bad: bool = False) -> dict[str, Any]:
    inputs = _build_inference_inputs(obs)
    output = _greedy_generate(inputs)
    completion = tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    try:
        return parse_json_action(completion)
    except Exception as exc:
        generation_stats["fallbacks"] += 1
        if len(generation_stats["parse_errors"]) < 5:
            generation_stats["parse_errors"].append({"error": str(exc), "completion": completion[:300]})
        if log_bad:
            print(f"bad generation, using fallback: {exc}")
            print("raw completion:", repr(completion[:300]))
        fallback = heuristic_action(obs)
        if fallback is None:
            fallback = {"action_type": "assistant_message", "assistant_message": "Could you confirm the details needed for scheduling?"}
        return fallback

def heuristic_policy(obs: dict[str, Any]) -> dict[str, Any]:
    action = heuristic_action(obs)
    if action is None:
        return {"action_type": "assistant_message", "assistant_message": "Could you confirm the details needed for scheduling?"}
    return action

async def evaluate_policy(policy_fn, label: str, scenarios=SCENARIOS, seeds=(123, 124), max_steps: int = 20, verbose: bool = False):
    rows = []
    for scenario_id in scenarios:
        for seed in seeds:
            before_fallbacks = generation_stats["fallbacks"] if "generation_stats" in globals() else 0
            async with FlatmateEndpoint() as env:
                obs = await env.reset(scenario_id, seed=seed)
                total_reward = 0.0
                steps = 0
                for step_idx in range(max_steps):
                    action = policy_fn(obs)
                    if verbose:
                        print(label, scenario_id, seed, step_idx, action)
                    obs = await env.step(action)
                    steps = step_idx + 1
                    total_reward += float(obs.get("reward") or obs.get("step_reward") or 0.0)
                    if obs.get("done"):
                        break
                after_fallbacks = generation_stats["fallbacks"] if "generation_stats" in globals() else before_fallbacks
                rows.append({
                    "policy": label,
                    "scenario_id": scenario_id,
                    "seed": seed,
                    "total_reward": total_reward,
                    "done": bool(obs.get("done")),
                    "bookings": len(obs.get("booked_visits", [])),
                    "violations": len(obs.get("violations", [])),
                    "steps": steps,
                    "fallbacks": after_fallbacks - before_fallbacks,
                })
    return rows

def raw_generate_action_text(obs: dict[str, Any]) -> str:
    inputs = _build_inference_inputs(obs)
    output = _greedy_generate(inputs)
    return tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

async def sanity_check_generations(limit: int = 4):
    rows = []
    for scenario_id in SCENARIOS[:limit]:
        async with FlatmateEndpoint() as env:
            obs = await env.reset(scenario_id, seed=321)
            raw = raw_generate_action_text(obs)
            try:
                parsed = parse_json_action(raw)
                ok = True
            except Exception as exc:
                parsed = str(exc)
                ok = False
            rows.append({"scenario_id": scenario_id, "json_ok": ok, "raw": raw[:240], "parsed_or_error": parsed})
    return pd.DataFrame(rows)

async def run_inference_each_task(policy_fn, label: str, seed: int = 321, max_steps: int = 20):
    rows = []
    for scenario_id in SCENARIOS:
        print(f"\n=== {label}: {scenario_id} ===")
        async with FlatmateEndpoint() as env:
            obs = await env.reset(scenario_id, seed=seed)
            total_reward = 0.0
            steps = 0
            before_fallbacks = generation_stats["fallbacks"]
            for step_idx in range(max_steps):
                action = policy_fn(obs)
                print(f"step={step_idx:02d} action={action}")
                obs = await env.step(action)
                steps = step_idx + 1
                total_reward += float(obs.get("reward") or obs.get("step_reward") or 0.0)
                if obs.get("done"):
                    break
            result = {
                "policy": label,
                "scenario_id": scenario_id,
                "seed": seed,
                "total_reward": total_reward,
                "done": bool(obs.get("done")),
                "bookings": len(obs.get("booked_visits", [])),
                "violations": len(obs.get("violations", [])),
                "steps": steps,
                "fallbacks": generation_stats["fallbacks"] - before_fallbacks,
            }
            print("result=", result)
            rows.append(result)
    return pd.DataFrame(rows)

generation_stats = {"fallbacks": 0, "parse_errors": []}
active_model = base_model_for_eval
base_generation_sanity_df = await sanity_check_generations()
base_generation_sanity_df["model"] = "base"
base_per_task_inference_df = await run_inference_each_task(safe_model_action, "base_model")
base_model_eval = await evaluate_policy(safe_model_action, "base_model")
base_stats = dict(generation_stats)

active_model = loaded_model_for_eval
generation_stats = {"fallbacks": 0, "parse_errors": []}
loaded_generation_sanity_df = await sanity_check_generations()
loaded_generation_sanity_df["model"] = "grpo_loaded"
loaded_per_task_inference_df = await run_inference_each_task(safe_model_action, "grpo_loaded")
per_task_inference_df = pd.concat([base_per_task_inference_df, loaded_per_task_inference_df], ignore_index=True)
loaded_stats = dict(generation_stats)

generation_sanity_df = pd.concat([base_generation_sanity_df, loaded_generation_sanity_df], ignore_index=True)
print("base_generation_stats", base_stats)
print("loaded_generation_stats", loaded_stats)
if loaded_stats["fallbacks"] > 0:
    print("WARNING: loaded fine-tuned model produced malformed JSON and used fallback.")
    print("First parse errors from the trained model:")
    for record in loaded_stats["parse_errors"]:
        print(" -", record["error"])
        print("   raw:", record["completion"][:200].replace("\n", " "))

heuristic_eval = await evaluate_policy(heuristic_policy, "heuristic")
active_model = loaded_model_for_eval
grpo_eval = await evaluate_policy(safe_model_action, "grpo_loaded")
eval_rows = heuristic_eval + base_model_eval + grpo_eval
eval_df = pd.DataFrame(eval_rows)
eval_df

## Performance Comparison

Compare heuristic rollout behavior against the trained GRPO policy on the same scenarios and seeds.

In [ ]:
def plot_policy_comparison(eval_df, title: str = "Base vs GRPO loaded-model comparison"):
    if eval_df is None or eval_df.empty or "policy" not in eval_df.columns:
        print("eval_df is empty because loaded-model generation used fallback or evaluation was skipped.")
        if "per_task_inference_df" in globals() and not per_task_inference_df.empty:
            print("Plotting per_task_inference_df instead. This is loaded-model rollout only, not heuristic-vs-model comparison.")
            display(per_task_inference_df)
            ax = per_task_inference_df.set_index("scenario_id")[["total_reward", "bookings", "violations", "fallbacks"]].plot(
                kind="bar",
                subplots=True,
                layout=(2, 2),
                figsize=(10, 7),
                legend=False,
                title="GRPO loaded-model per-task inference",
            )
            for axis in ax.ravel():
                axis.grid(axis="y", alpha=0.3)
                axis.set_xlabel("")
            plt.tight_layout()
            plt.show()
        return pd.DataFrame()

    summary = (
        eval_df.groupby("policy", as_index=True)
        .agg(
            avg_reward=("total_reward", "mean"),
            completion_rate=("done", "mean"),
            avg_bookings=("bookings", "mean"),
            avg_violations=("violations", "mean"),
            avg_steps=("steps", "mean"),
            avg_fallbacks=("fallbacks", "mean") if "fallbacks" in eval_df.columns else ("steps", "size"),
        )
        .sort_index()
    )
    plot_cols = ["avg_reward", "completion_rate", "avg_bookings", "avg_violations"]
    if "avg_fallbacks" in summary.columns:
        plot_cols.append("avg_fallbacks")
    axes = summary[plot_cols].plot(
        kind="bar",
        subplots=True,
        layout=(3, 2),
        figsize=(10, 9),
        legend=False,
        title=title,
    )
    for ax in axes.ravel():
        ax.grid(axis="y", alpha=0.3)
        ax.set_xlabel("")
    plt.tight_layout()
    plt.show()
    return summary

comparison_summary = plot_policy_comparison(eval_df)
comparison_summary

In [ ]:
# Optional Hub upload after training.
# from huggingface_hub import notebook_login
# notebook_login()
# trainer.push_to_hub("flatmate-rl-grpo-policy")